# Phase 4 — Class Imbalance & Sampling

**Objective:** Quantify the churn imbalance and choose a defensible strategy for handling it.

Loads `features_transformed.parquet`, joins it with the competition labels and test
accounts, builds the stratified train/validation split, and saves `train_val_split.parquet`
+ `test_features.parquet` for the modeling notebooks.

This pipeline's chosen strategy is **`is_unbalance=True`** in LightGBM (equivalent to
class weighting — it auto-scales the gradient contribution of the minority class).
The comparison cell below documents why this was chosen over SMOTE / manual undersampling.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split

USER_COL = "ACCOUNT_ID"

transformed_df2 = pl.read_parquet("exp_files/features_transformed.parquet")
print(f"Loaded features_transformed_df2 — shape: {transformed_df2.shape}")

feature_columns = transformed_df2.columns[1:]
print("Feature columns:", feature_columns)

Loaded features_transformed_df2 — shape: (850000, 37)
Feature columns: ['balance_std_dev_last30D', 'balance_std_dev_last14D', 'balance_std_dev_last7D', 'balance_trend', 'bill_pay_count_last_30D', 'bill_pay_count_last_14D', 'bill_pay_count_last_7D', 'merchant_count_last_30D', 'merchant_count_last_14D', 'merchant_count_last_7D', 'p2p_count_last_30D', 'p2p_count_last_14D', 'p2p_count_last_7D', 'cash_in_amount_last_30D', 'cash_in_count_last_14D', 'cash_in_count_last_7D', 'cash_out_count_last_30D', 'cash_out_count_last_14D', 'cash_out_count_last_7D', 'bill_pay_ratio', 'unique_trx_types_used', 'total_trx_amount', 'frequency', 'trx_amount_last_30D', 'freq_last_30D', 'recency', 'average_gap_days', 'trx_trend', 'consistency_metric', 'p2p_ratio_90d', 'importance', 'freq_last_14D', 'freq_last_7D', 'velocity_last_14d', 'velocity_last_7d', 'inactivity_multiplier']


## Join labels and test accounts

In [2]:
train_users = pl.read_csv("/media/tasfreak/Study/bkash-presents-nsucec-datathon/public/train_labels.csv")
test_users = pl.read_csv("/media/tasfreak/Study/bkash-presents-nsucec-datathon/public/test.csv")

train_df = (train_users.join(transformed_df2, on=USER_COL, how="left"))
test_df = (test_users.join(transformed_df2, on=USER_COL, how="left"))
test_df.shape

(255000, 37)

In [3]:
train_df.head(10)

ACCOUNT_ID,CHURN,balance_std_dev_last30D,balance_std_dev_last14D,balance_std_dev_last7D,balance_trend,bill_pay_count_last_30D,bill_pay_count_last_14D,bill_pay_count_last_7D,merchant_count_last_30D,merchant_count_last_14D,merchant_count_last_7D,p2p_count_last_30D,p2p_count_last_14D,p2p_count_last_7D,cash_in_amount_last_30D,cash_in_count_last_14D,cash_in_count_last_7D,cash_out_count_last_30D,cash_out_count_last_14D,cash_out_count_last_7D,bill_pay_ratio,unique_trx_types_used,total_trx_amount,frequency,trx_amount_last_30D,freq_last_30D,recency,average_gap_days,trx_trend,consistency_metric,p2p_ratio_90d,importance,freq_last_14D,freq_last_7D,velocity_last_14d,velocity_last_7d,inactivity_multiplier
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""CUST00109362""",0,8.370291,7.72265,7.902786,0.658381,1.94591,1.791759,1.386294,0.0,0.0,0.0,0.0,0.0,0.0,9.637215,1.791759,1.386294,1.386294,1.386294,1.386294,0.256234,1.386294,11.317978,3.871201,9.880177,2.772589,0.0,0.898746,0.725937,0.806108,0.0,0.110169,2.639057,2.302585,0.239673,0.17185,0.0
"""CUST00480386""",0,10.97503,10.676394,10.279175,1.258961,1.94591,1.609438,1.098612,3.178054,2.397895,2.079442,0.693147,0.0,0.0,12.258062,2.302585,1.386294,0.693147,0.0,0.0,0.057439,1.791759,13.729506,4.905275,13.183376,3.871201,0.693147,0.262942,0.714653,0.965026,0.000383,0.832358,3.178054,2.564949,0.15732,0.085158,1.251422
"""CUST00637759""",0,9.16382,9.423996,9.813539,0.833043,0.0,0.0,0.0,0.0,0.0,0.0,1.386294,0.693147,0.693147,9.506321,1.609438,1.386294,1.94591,0.693147,0.693147,0.0,1.386294,11.910692,3.688879,11.147554,2.833213,0.0,0.997143,0.836248,0.774381,0.097763,0.191179,1.94591,1.791759,0.139762,0.117783,0.0
"""CUST00303030""",0,1.773819,1.754168,1.744044,0.002053,0.0,0.0,0.0,1.609438,1.098612,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.693147,10.653649,2.484907,7.769184,1.609438,2.197225,1.94591,0.693147,0.702793,0.0,0.058208,1.098612,0.0,0.154151,0.0,0.837886
"""CUST00395836""",0,1.378377,1.08713,0.932856,0.083333,0.0,0.0,0.0,0.0,0.0,0.0,2.772589,2.079442,1.609438,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.693147,11.281672,3.526361,10.502007,2.70805,0.0,1.178655,0.860201,0.673293,0.693147,0.106443,2.079442,1.609438,0.187212,0.111226,0.0
"""CUST00603780""",0,1.909622,2.159803,2.435971,0.071343,1.098612,0.693147,0.693147,1.098612,0.693147,0.693147,1.098612,0.693147,0.693147,0.0,0.0,0.0,0.693147,0.0,0.0,0.244154,1.791759,10.893313,3.526361,9.377565,2.079442,0.0,1.197703,0.587787,0.801911,0.094115,0.073406,1.386294,1.386294,0.084557,0.084557,0.0
"""CUST00806746""",0,0.913626,1.022088,1.282093,0.015737,0.0,0.0,0.0,0.0,0.0,0.0,1.098612,0.693147,0.0,0.0,0.0,0.0,3.806662,3.178054,2.772589,0.005255,1.609438,13.1696,4.912655,11.759587,3.828641,0.693147,0.249655,0.682452,0.937445,0.009575,0.554986,3.218876,2.772589,0.162519,0.104625,1.282877
"""CUST00591646""",0,0.870355,0.819436,0.81308,0.001445,2.397895,1.94591,1.609438,2.197225,1.609438,1.386294,3.218876,2.484907,1.791759,5.442331,0.693147,0.0,0.0,0.0,0.0,0.073563,1.609438,13.417732,4.990433,12.421447,3.78419,0.0,0.216223,0.572069,0.93785,0.503993,0.668271,3.135494,2.564949,0.139466,0.078472,0.0
"""CUST00363357""",0,7.546348,2.627793,1.545419,0.11061,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.533231,0.693147,0.693147,1.386294,0.693147,0.0,0.0,1.386294,10.108613,2.397895,9.121045,1.791759,1.94591,2.268684,0.788457,0.589198,0.000407,0.034161,1.098612,0.693147,0.167054,0.087011,0.521416


## Quantify the imbalance

A model that always predicts "no churn" still scores high on raw accuracy here —
which is exactly why accuracy is the wrong metric for this problem (Phase 5 uses
AUC / Precision@K / Recall@K instead).

In [4]:
churn_rate = train_df["CHURN"].mean()
print(f"Churn rate: {churn_rate:.2%}")
print(f"A model predicting 'never churn' on this data would score "
      f"{1 - churn_rate:.2%} accuracy while being completely useless.")

Churn rate: 12.68%
A model predicting 'never churn' on this data would score 87.32% accuracy while being completely useless.


## Strategy comparison

| Strategy | How it works | Trade-off |
|---|---|---|
| `is_unbalance=True` (chosen) | LightGBM auto-scales minority-class gradient — equivalent to class weighting | No synthetic data, no row duplication, fastest to train |
| SMOTE | Generates synthetic minority-class rows by interpolating between nearest neighbors | Can introduce unrealistic feature combinations on financial ratio features |
| Random undersampling | Drops majority-class rows until balanced | Throws away real data — risky at this scale where majority-class patterns still matter |

**Chosen approach: `is_unbalance=True`.** This was selected because it requires no
data modification (synthetic or dropped), trains in the same time as the unweighted
baseline, and lets LightGBM's native loss function rebalance gradient contributions —
which directly targets the FN (missed churners) vs FP (false alarms) trade-off described
in the rubric, since increasing minority-class weight pushes recall up at a controlled
precision cost.

## Build the stratified train/validation split

In [5]:
X = train_df.select(feature_columns).to_pandas()
y = train_df["CHURN"].to_pandas()

# Perform a Stratified 80/20 Split
X_train, X_val, y_train, y_val = train_test_split(
    X, 
    y, 
    test_size=0.20,      # 20% validation data
    random_state=42,     # Ensures reproducible splits
    stratify=y           # Keeps churn ratio identical in train and val subsets
)

print(f"Training Features Shape: {X_train.shape} | Validation Features Shape: {X_val.shape}")

Training Features Shape: (476000, 36) | Validation Features Shape: (119000, 36)


## Save splits for downstream notebooks

In [6]:
X_train.to_parquet("exp_files/X_train.parquet")
X_val.to_parquet("exp_files/X_val.parquet")
y_train.to_frame("CHURN").to_parquet("exp_files/y_train.parquet")
y_val.to_frame("CHURN").to_parquet("exp_files/y_val.parquet")
test_df.write_parquet("exp_files/test_df.parquet")
test_users.write_parquet("exp_files/test_users.parquet")

print("Saved: X_train.parquet, X_val.parquet, y_train.parquet, y_val.parquet, test_df.parquet, test_users.parquet")

Saved: X_train.parquet, X_val.parquet, y_train.parquet, y_val.parquet, test_df.parquet, test_users.parquet


## Next notebook

Continue to **`05_model_training.ipynb`**, which loads these splits and trains
the LightGBM baseline model.